# Colab CheXNet Workflow

This notebook bootstraps a separate Colab workspace for `chestxray14`.
It is meant for candidate discovery and staged follow-up, not for sharing a
ledger with an active local run.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Choose A Session Tag

Use a fresh session tag each time you start a new Colab lane.

In [ ]:
SESSION = 'chestxray14_colab_frontier_20260318'
WORKDIR = '/content/autoresearch-medimage-colab'
DRIVE_ROOT = '/content/drive/MyDrive/autoresearch-medimage-colab'
REPO_URL = 'https://github.com/mlungrengmail/autoresearch-medimage.git'

## 3. Clone The Repo

In [ ]:
!rm -rf $WORKDIR
!git clone $REPO_URL $WORKDIR

## 4. Install Dependencies

In [ ]:
%cd $WORKDIR
!python -m pip install -q uv
!uv sync
!nvidia-smi

## 5. Configure Kaggle Credentials

Upload `kaggle.json` to `MyDrive/kaggle.json` before running the next cell.

In [ ]:
!mkdir -p /root/.kaggle
!cp /content/drive/MyDrive/kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

## 6. Prepare The Dataset

In [ ]:
%cd $WORKDIR
!uv run prepare.py --dataset chestxray14

## 7. Optional Baseline

Run this if you want a single baseline row before using the queue or adaptive
lane.

In [ ]:
%cd $WORKDIR
!uv run train.py --recipe chexnet --session $SESSION --save-checkpoint > run.log 2>&1
!tail -n 120 run.log
!uv run python tools/mark_result.py --dataset chestxray14 --session $SESSION --status keep --append-note 'session baseline'

## 8. Choose One Colab Lane

Run exactly one of the next two cells in a fresh session tag.

- The frontier queue is the simpler, safer default.
- The adaptive lane keeps multiple branches alive and writes search metadata
  under `logs/<session>/search/`.

In [ ]:
%cd $WORKDIR
!uv run python tools/run_recipe_queue.py \
  --preset chestxray14_backbone_frontier \
  --dataset chestxray14 \
  --session $SESSION

In [ ]:
%cd $WORKDIR
!uv run python tools/run_search_session.py \
  --space chestxray14 \
  --session $SESSION

## 9. Sync Artifacts Back To Drive

Run this after a baseline, queue, or adaptive session so you can recover after
a Colab disconnect.

In [ ]:
!mkdir -p "$DRIVE_ROOT/results" "$DRIVE_ROOT/checkpoints" "$DRIVE_ROOT/artifacts" "$DRIVE_ROOT/logs"
!cp $WORKDIR/results.tsv "$DRIVE_ROOT/results/results.tsv" || true
!cp $WORKDIR/run.log "$DRIVE_ROOT/artifacts/run.log" || true
!cp -r $WORKDIR/checkpoints/. "$DRIVE_ROOT/checkpoints/" || true
!cp -r $WORKDIR/logs/. "$DRIVE_ROOT/logs/" || true

## 10. Next Step

For staged refinement, recovery after disconnects, and the agent prompt, use
`docs/COLAB_OVERNIGHT_RUN.md` in the cloned workspace.

If you are doing benchmark-style work, treat the short Colab search as a
candidate-discovery stage rather than as the final winner.